# 01 — Exploratory Data Analysis
**Cricket Analytics Platform** — Phase 3 Data Exploration

This notebook explores the match, player, and series data collected by the Celery pipeline.

**Run the data pipeline first:**
```bash
cd backend
python manage.py sync_matches
```

In [ ]:
import sys, os

# Bootstrap Django so we can query models directly
BACKEND = os.path.abspath("../../backend")
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)

os.environ["DJANGO_SETTINGS_MODULE"] = "config.settings.dev"

import django
django.setup()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid", palette="viridis")
plt.rcParams["figure.figsize"] = (12, 5)

print("✅ Django setup complete. Ready to query models.")

## 1. Database Overview

In [ ]:
from apps.matches.models import Match, Team, Venue
from apps.players.models import Player, PlayerMatchStats
from apps.series.models import Series
from apps.predictions.models import PredictionJob

stats = {
    "Total Matches"       : Match.objects.count(),
    "  Live"              : Match.objects.filter(status='live').count(),
    "  Upcoming"          : Match.objects.filter(status='upcoming').count(),
    "  Completed"         : Match.objects.filter(status='complete').count(),
    "Teams"               : Team.objects.count(),
    "Venues"              : Venue.objects.count(),
    "Players"             : Player.objects.count(),
    "Player Match Stats"  : PlayerMatchStats.objects.count(),
    "Series"              : Series.objects.count(),
    "Prediction Jobs"     : PredictionJob.objects.count(),
}

for k, v in stats.items():
    print(f"{k:<25}: {v}")

## 2. Match Distribution by Format & Category

In [ ]:
from django.db.models import Count

fmt_qs = Match.objects.values('format').annotate(n=Count('id')).order_by('-n')
cat_qs = Match.objects.values('category').annotate(n=Count('id')).order_by('-n')

df_fmt = pd.DataFrame(fmt_qs)
df_cat = pd.DataFrame(cat_qs)

if not df_fmt.empty:
    fig, axes = plt.subplots(1, 2)
    df_fmt.plot.bar(ax=axes[0], x='format', y='n', legend=False, color='#00d4aa')
    axes[0].set_title('Matches by Format'); axes[0].set_xlabel('')
    
    df_cat.plot.pie(ax=axes[1], y='n', labels=df_cat['category'].tolist(),
                   autopct='%1.0f%%', legend=False)
    axes[1].set_title('Matches by Category'); axes[1].set_ylabel('')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️  No match data yet. Run: python manage.py sync_matches")

## 3. Match Timeline

In [ ]:
matches_qs = Match.objects.filter(
    match_date__isnull=False
).values('match_date', 'format', 'status').order_by('match_date')

df_m = pd.DataFrame(list(matches_qs))

if not df_m.empty:
    df_m['match_date'] = pd.to_datetime(df_m['match_date'])
    df_m['month'] = df_m['match_date'].dt.to_period('M').astype(str)
    monthly = df_m.groupby('month').size()
    monthly.plot.bar(color='#f59e0b', figsize=(14,4))
    plt.title('Matches per Month')
    plt.xlabel('Month'); plt.ylabel('Count')
    plt.tight_layout(); plt.show()
else:
    print("No data yet.")

## 4. Top Teams by Match Count

In [ ]:
from django.db.models import Q

teams = Team.objects.annotate(
    total_matches=Count('home_matches') + Count('away_matches')
).order_by('-total_matches')[:20]

df_t = pd.DataFrame(list(teams.values('name', 'total_matches')))

if not df_t.empty:
    df_t.plot.barh(x='name', y='total_matches', color='#00d4aa', legend=False)
    plt.title('Top 20 Teams by Match Count')
    plt.xlabel('Matches')
    plt.gca().invert_yaxis()
    plt.tight_layout(); plt.show()
else:
    print("No team data yet.")

## 5. Pre-Match Feature Preview (Phase 3)

Once we have completed match data, we'll build the ML feature matrix here.

In [ ]:
# Preview the feature engineering pipeline
# Uncomment after Phase 1 data collection

# from src.utils.data_loader import load_matches_df
# from src.utils.preprocessor import encode_features

# df = load_matches_df()
# print(f"Loaded {len(df)} completed matches for training")
# df_enc = encode_features(df)
# print(df_enc[['format_enc','category_enc','pitch_enc','target']].describe())

print("Feature engineering notebook ready for Phase 3.")
print("Run Phase 1 data pipeline first to populate the DB.")